# Imports, path, data

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import pykitti
from pathlib import Path
from datetime import datetime

base = Path.home() / 'SensorTrust' / 'datasets' / 'kitti'

data = pykitti.raw(
    base_path=str(base),
    date='2011_09_26',
    drive='0009'
)

print(f"Loaded: {len(data.oxts)} GPS/IMU, {len(data.cam2_files)} Camera, {len(data.velo_files)} LiDAR")

### `data.oxts` : GPS/IMU navigation data packets (tracking position, speed, and orientation).
### `data.cam2_files`: Image files from the left color camera.
### `data.velo_files` :  Raw 3D point cloud files from the Velodyne LiDAR scanner.

In [ ]:
vfs = [] #list to store forward velocity
for frame in data.oxts:
    vfs.append(frame.packet.vf)

plt.figure(figsize=(10, 4))
plt.plot(vfs)
plt.xlabel('Frame')
plt.ylabel('Forward Velocity (m/s)')
plt.title('Vehicle Speed Over Time')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Speed range: {min(vfs):.1f} to {max(vfs):.1f} m/s")
print(f"Vehicle was stationary at start: {vfs[0] < 0.1}")

### A frame is a single, frozen snapshot of data.It is a physical file or entry (1 image + 1 LiDAR scan + 1 GPS reading).
### 

In [ ]:
# ============================================================
# 2. Extract IMU acceleration and yaw rate
# ============================================================
axs = []
wzs = []
for frame in data.oxts:
    axs.append(frame.packet.ax)
    wzs.append(frame.packet.wz)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))

ax1.plot(axs)
ax1.set_ylabel('Forward Accel (m/s²)')
ax1.grid(True, alpha=0.3)

ax2.plot(wzs)
ax2.set_ylabel('Yaw Rate (rad/s)')
ax2.set_xlabel('Frame')
ax2.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Print out the exact timestamps for the first 5 frames
for i in range(5):
    print(f"Frame {i}: {data.timestamps[i]}")

# Calculate the actual time gaps between consecutive frames
gaps = []
for i in range(len(data.timestamps) - 1):
    time_difference = (data.timestamps[i+1] - data.timestamps[i]).total_seconds()
    gaps.append(time_difference)

print(f"\nAverage gap between sensors: {sum(gaps)/len(gaps):.4f} seconds")


# 3. Check LiDAR point counts

In [ ]:
point_counts = []
for i in range(len(data.velo_files)):
    scan = data.get_velo(i)
    point_counts.append(scan.shape[0])

print(f"LiDAR points per scan: min={min(point_counts)}, max={max(point_counts)}, mean={np.mean(point_counts):.0f}")

In [ ]:

# ============================================================
# 4. Check camera image dimensions
# ============================================================
cam0_pil = data.get_cam2(0)
cam0 = np.array(cam0_pil)

print(f"Camera image shape: {cam0.shape} (height, width, channels)")
print(f"Camera pixel range: {cam0.min()} to {cam0.max()}")
print(f"Camera dtype: {cam0.dtype}")

In [ ]:
# 5. Timestamp alignment check
def load_timestamps(filepath):
    timestamps = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            date_part, time_part = line.split(' ')
            whole_secs, frac = time_part.split('.')
            frac = frac[:6]
            dt = datetime.strptime(f"{date_part} {whole_secs}.{frac}", '%Y-%m-%d %H:%M:%S.%f')
            timestamps.append(dt.timestamp())
    return np.array(timestamps)
sync_base = Path.home() / 'SensorTrust/datasets/kitti/2011_09_26/2011_09_26_drive_0009_sync'
oxts_t = load_timestamps(str(sync_base/ 'oxts/timestamps.txt'))
velo_t = load_timestamps(str(sync_base/ 'velodyne_points/timestamps.txt'))
cam_t  = load_timestamps(str(sync_base/ 'image_02/timestamps.txt'))

print(f"\nSensor frequencies:")
print(f"  GPS/IMU: {1.0/np.mean(np.diff(oxts_t)):.1f} Hz")
print(f"  LiDAR:   {1.0/np.mean(np.diff(velo_t)):.1f} Hz")
print(f"  Camera:  {1.0/np.mean(np.diff(cam_t)):.1f} Hz")

print(f"\nFirst timestamp offsets (relative to GPS):")
print(f"  LiDAR starts:  {(velo_t[0] - oxts_t[0])*1000:.1f} ms after GPS")
print(f"  Camera starts: {(cam_t[0] - oxts_t[0])*1000:.1f} ms after GPS")

In [ ]:
stationary_frames = []
for i, frame in enumerate(data.oxts):
    if abs(frame.packet.vf) < 0.1:
        stationary_frames.append(i)

print(f"Stationary frames: {len(stationary_frames)} out of {len(data.oxts)}")
print(f"First stationary frame: {stationary_frames[0] if stationary_frames else 'None'}")
print(f"Stationary segments: ", end="")

# Find consecutive runs
if stationary_frames:
    runs = []
    start = stationary_frames[0]
    for i in range(1, len(stationary_frames)):
        if stationary_frames[i] != stationary_frames[i-1] + 1:
            runs.append((start, stationary_frames[i-1]))
            start = stationary_frames[i]
    runs.append((start, stationary_frames[-1]))
    
    for s, e in runs:
        print(f"Frames {s}-{e} ({e-s+1} frames)", end="  ")